In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
import random

In [17]:
ratings=pd.read_csv('ratings.csv')
movies=pd.read_csv('movies.csv')
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [18]:
user_movie_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)
user_movie_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
user_similarity=cosine_similarity(user_movie_matrix)
user_similarity_df=pd.DataFrame(user_similarity,index=user_movie_matrix.index,columns=user_movie_matrix.index)
user_similarity_df.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.027283,0.059720,0.194395,0.129080,0.128152,0.158744,0.136968,0.064263,0.016875,...,0.080554,0.164455,0.221486,0.070669,0.153625,0.164191,0.269389,0.291097,0.093572,0.145321
2,0.027283,1.000000,0.000000,0.003726,0.016614,0.025333,0.027585,0.027257,0.000000,0.067445,...,0.202671,0.016866,0.011997,0.000000,0.000000,0.028429,0.012948,0.046211,0.027565,0.102427
3,0.059720,0.000000,1.000000,0.002251,0.005020,0.003936,0.000000,0.004941,0.000000,0.000000,...,0.005048,0.004892,0.024992,0.000000,0.010694,0.012993,0.019247,0.021128,0.000000,0.032119
4,0.194395,0.003726,0.002251,1.000000,0.128659,0.088491,0.115120,0.062969,0.011361,0.031163,...,0.085938,0.128273,0.307973,0.052985,0.084584,0.200395,0.131746,0.149858,0.032198,0.107683
5,0.129080,0.016614,0.005020,0.128659,1.000000,0.300349,0.108342,0.429075,0.000000,0.030611,...,0.068048,0.418747,0.110148,0.258773,0.148758,0.106435,0.152866,0.135535,0.261232,0.060792


In [30]:
def recommend_movies(user_id,top_n=5):
    similar_users = user_similarity_df[user_id].sort_values(ascending=False).index[1:top_n+1]
    similar_users_ratings = user_movie_matrix.loc[similar_users]
    mean_ratings=similar_users_ratings.mean(axis=0)
    user_ratings=user_movie_matrix.loc[user_id]
    mean_ratings=user_ratings[user_ratings==0]
    recommend_movie_ids=mean_ratings.sort_values(ascending=False).index[:top_n]
    recommended_movies=movies[movies['movieId'].isin(recommend_movie_ids)]
    return recommended_movies.sort_values(by='movieId',ascending=False)


In [ ]:
def explore_new_movies(user_id, top_n=5):
    user_seen = user_movie_matrix.loc[user_id]
    seen_movies = user_seen[user_seen > 0]
    unseen_movies = movies[~movies['movieId'].isin(seen_movies)]
    return unseen_movies.sample(top_n)

In [37]:
def trending_movies(top_n=5):
    top_movie_ids = ratings.groupby('movieId').size().sort_values(ascending=False).head(top_n).index
    return movies[movies['movieId'].isin(top_movie_ids)]

In [38]:
def display(user_id):
    recs=recommend_movies(user_id,top_n=5)
    print("\n MOVIE RECOMMENDATION AGENT\n")
    print("User ID:",user_id)
    print("\n Today's Top Picks for You:\n ")
    for title in recs['title']:
        print("- ",title)
    print("\nExplore New Movies For You:\n")
    new_movies = explore_new_movies(user_id)
    for title in new_movies['title']:
         print("-", title)
    print("\n📈 Trending Movies")
    trending = trending_movies()
    for title in trending['title']:
        print("-", title)
    user_rated=user_movie_matrix.loc[user_id]
    rated_ids=user_rated[user_rated>0].index

In [39]:
display(1)
display(10)


 MOVIE RECOMMENDATION AGENT

User ID: 1

 Today's Top Picks for You:
 
-  Tom and Huck (1995)
-  Sabrina (1995)
-  Father of the Bride Part II (1995)
-  Waiting to Exhale (1995)
-  Jumanji (1995)

Explore New Movies For You:

- Plastic (2014)
- Maze Runner, The (2014)
- Wayne's World (1992)
- Ring of Terror (1962)
- The Expendables 3 (2014)

📈 Trending Movies
- Pulp Fiction (1994)
- Shawshank Redemption, The (1994)
- Forrest Gump (1994)
- Silence of the Lambs, The (1991)
- Matrix, The (1999)

 MOVIE RECOMMENDATION AGENT

User ID: 10

 Today's Top Picks for You:
 
-  Father of the Bride Part II (1995)
-  Waiting to Exhale (1995)
-  Grumpier Old Men (1995)
-  Jumanji (1995)
-  Toy Story (1995)

Explore New Movies For You:

- Suburban Commando (1991)
- In the Line of Fire (1993)
- Expendables 2, The (2012)
- Life with Mikey (1993)
- Opposite of Sex, The (1998)

📈 Trending Movies
- Pulp Fiction (1994)
- Shawshank Redemption, The (1994)
- Forrest Gump (1994)
- Silence of the Lambs, The (19

In [40]:
import streamlit as st

In [ ]:
import streamlit as st
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import random

# ---------------- DATA ----------------
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")

user_movie_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

user_similarity = cosine_similarity(user_movie_matrix)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

# ---------------- FUNCTIONS ----------------
def trending_movies():
    top_ids = ratings.groupby('movieId').size().sort_values(ascending=False).head(5).index
    return movies[movies['movieId'].isin(top_ids)]

def explore_new(user_id):
    seen = user_movie_matrix.loc[user_id]
    seen_movies = seen[seen > 0].index
    unseen = movies[~movies['movieId'].isin(seen_movies)]
    return unseen.sample(5)

def recommend(user_id):
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:6]
    similar_ratings = user_movie_matrix.loc[similar_users]
    mean_ratings = similar_ratings.mean(axis=0)

    user_seen = user_movie_matrix.loc[user_id]
    mean_ratings = mean_ratings[user_seen == 0]

    top_ids = mean_ratings.sort_values(ascending=False).head(10).index

    recs = movies[movies['movieId'].isin(top_ids)]
    return recs

# ---------------- UI ----------------
st.title("🎬 Mini AI Recommender")

user_id = st.number_input("Enter User ID", min_value=1, max_value=1000, step=1)

if st.button("Get Recommendations"):

    st.subheader("🔥 Top Picks For You")
    recs = recommend(user_id)
    st.write(recs[['title']].head(5))

    st.subheader("📈 Trending Now")
    st.write(trending_movies()[['title']])

    st.subheader("🌍 Explore New Movies")
    st.write(explore_new(user_id)[['title']])

2026-04-11 17:20:06.195 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 17:20:07.060 
  command:

    streamlit run c:\Users\SRIHARSHITH\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-04-11 17:20:07.064 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 17:20:07.066 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 17:20:07.067 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 17:20:07.068 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 17:20:07.070 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 

: 